# 06 - Hyperparameter tuning with GridSearchCV
We tune three models on the **training set only** (`<= 2021`): **Random Forest**, **XGBoost**, and **CatBoost**.
We chose XGBoost over scikit-learn's GradientBoostingRegressor because it is faster for grid search and is one of the three base models in our Voting/Stacking ensembles.

**Rules we follow:**
- `cv = TimeSeriesSplit(5)` (with the 1-week gap from step 03) so validation is always in the future of training.
- `scoring = "neg_root_mean_squared_error"` on `target_log` (since models train on `log1p(cases)`).
- `n_jobs = -1`, `refit = True`, `random_state = 42` everywhere.
- The test set (`>= 2022`) is **never** touched during tuning; each best model is evaluated on it **once** after the search.

In [1]:
# AI-GENERATED (Claude via Antigravity)
import json, os, sys, time, warnings
import numpy as np, pandas as pd
sys.path.append("../src")
from split import *
from evaluate import regression_scores, to_cases
from stacking import CatBoostDistrict
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
warnings.filterwarnings("ignore")

df = load_features("../data/processed/features.csv")
train, test = time_split(df)
features = feature_columns(df)
X_train, X_test = train[features], test[features]
y_train_log, y_train_raw, y_test_raw = train[TARGET_LOG], train[TARGET_RAW], test[TARGET_RAW]
cv = make_cv(train)
N_CPUS = os.cpu_count() or 4
print(f"Train rows: {len(X_train):,} | Test rows: {len(X_test):,} | CV splits: {cv.n_splits} | CPU cores: {N_CPUS}")

Train rows: 19,400 | Test rows: 5,150 | CV splits: 5 | CPU cores: 16


In [2]:
# AI-GENERATED (Claude via Antigravity)
def estimate_and_tune(name, family, estimator, param_grid, sample_est):
    """Estimate runtime from 1 single fit, run GridSearchCV on train only, show top 5, then score ONCE on test."""
    n_combos = int(np.prod([len(v) for v in param_grid.values()]))
    t0 = time.perf_counter()
    sample_est.fit(X_train, y_train_log)
    one_fit = time.perf_counter() - t0
    # Each TimeSeriesSplit fold uses ~55% of training rows on average; GridSearchCV runs folds in parallel
    est_sec = (one_fit * 0.55 * n_combos * cv.n_splits) / max(1, min(N_CPUS, 4))
    print(f"[{name}] {n_combos} combinations x {cv.n_splits} folds = {n_combos * cv.n_splits} fits.")
    print(f"Estimated runtime: ~{est_sec:.0f}s (1 full fit took {one_fit:.1f}s)")

    t1 = time.perf_counter()
    gs = GridSearchCV(estimator, param_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1, refit=True)
    gs.fit(X_train, y_train_log)
    elapsed = time.perf_counter() - t1

    cv_df = pd.DataFrame(gs.cv_results_).sort_values("rank_test_score")
    top5 = cv_df[["rank_test_score", "mean_test_score", "std_test_score"] + [f"param_{k}" for k in param_grid]].head(5)
    top5 = top5.rename(columns={"mean_test_score": "mean_CV_neg_RMSE_log", "std_test_score": "std_CV"})
    display(top5)

    # Check if any winning hyperparameter sits at the boundary of its candidate list
    edges = []
    for k, vals in param_grid.items():
        if len(vals) > 1 and gs.best_params_[k] in (vals[0], vals[-1]):
            edges.append(f"{k}={gs.best_params_[k]} (edge of {vals})")
    print("Best params:", gs.best_params_)
    print("At grid edge?:", "; ".join(edges) if edges else "No, all interior")

    # Evaluate ONCE on the held-out test set (original case scale)
    tr = regression_scores(y_train_raw, to_cases(gs.predict(X_train)))
    te = regression_scores(y_test_raw, to_cases(gs.predict(X_test)))
    row = {"model": f"{name} (tuned)", "family": family, "train_RMSE": tr["RMSE"], "test_RMSE": te["RMSE"],
           "test_MAE": te["MAE"], "test_R2": te["R2"], "train_seconds": round(elapsed, 2)}
    print(f"Done in {elapsed:.1f}s -> train RMSE: {tr['RMSE']:.2f} | test RMSE: {te['RMSE']:.2f} | test MAE: {te['MAE']:.2f} | R2: {te['R2']:.3f}")
    return gs, row

tuned_rows, best_params = [], {}

## 1. Random Forest (Bagging)
In step 04, default Random Forest overfit heavily (`train RMSE 11.5` vs `test RMSE 27.8`). We fix `n_estimators=200` and tune the three parameters that control overfitting and tree diversity.

| parameter | candidate values | why this range |
|---|---|---|
| `max_depth` | `[8, 14, None]` | Default `None` grows pure leaves that memorise noise (high variance); capping depth at `8` or `14` trades a little bias for much lower variance. |
| `min_samples_leaf` | `[3, 10]` | Requiring at least `3` or `10` weeks per leaf stops the tree from fitting single outbreak spikes (lower variance, slightly higher bias). |
| `max_features` | `["sqrt", 0.5]` | Fewer features per split (`sqrt`) decorrelates trees (lower variance), while `0.5` ensures strong predictors like `cases_lag_0` appear often (lower bias). |

Total combinations: `3 x 2 x 2 = 12` (well under 36).

In [3]:
# AI-GENERATED (Claude via Antigravity)
rf_pipe = Pipeline([("prep", make_preprocessor(features)),
                    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1))])
rf_grid = {
    "model__max_depth": [8, 14, None],
    "model__min_samples_leaf": [3, 10],
    "model__max_features": ["sqrt", 0.5],
}
rf_sample = Pipeline([("prep", make_preprocessor(features)),
                      ("model", RandomForestRegressor(n_estimators=200, max_depth=14, random_state=42, n_jobs=-1))])
gs_rf, row_rf = estimate_and_tune("Random Forest", "bagging", rf_pipe, rf_grid, rf_sample)
tuned_rows.append(row_rf); best_params["Random Forest"] = gs_rf.best_params_

[Random Forest] 12 combinations x 5 folds = 60 fits.
Estimated runtime: ~179s (1 full fit took 21.6s)


,rank_test_score,mean_CV_neg_RMSE_log,std_CV,param_model__max_depth,param_model__min_samples_leaf,param_model__max_features
3,1,-0.627971,0.060200,8,10,0.5
7,2,-0.628092,0.060400,14,10,0.5
2,3,-0.628292,0.058708,8,3,0.5
11,4,-0.628349,0.059663,None,10,0.5
10,5,-0.631086,0.056893,None,3,0.5


Best params: {'model__max_depth': 8, 'model__max_features': 0.5, 'model__min_samples_leaf': 10}
At grid edge?: model__max_depth=8 (edge of [8, 14, None]); model__min_samples_leaf=10 (edge of [3, 10]); model__max_features=0.5 (edge of ['sqrt', 0.5])


Done in 229.7s -> train RMSE: 26.62 | test RMSE: 25.79 | test MAE: 11.95 | R2: 0.831


## 2. XGBoost (Boosting)
Default XGBoost had `train RMSE 19.0` vs `test RMSE 28.3`. We fix `subsample=0.8` (to keep variance down) and tune tree count, depth, and learning rate.

| parameter | candidate values | why this range |
|---|---|---|
| `n_estimators` | `[200, 400]` | More boosting rounds lower bias, especially when paired with a smaller learning rate, though too many can overfit. |
| `max_depth` | `[3, 5, 7]` | Shallow trees (`3`) restrict interactions and keep variance low; deeper trees (`7`) capture complex climate-case patterns (lower bias, higher variance). |
| `learning_rate` | `[0.03, 0.1]` | Smaller steps (`0.03`) shrink each tree's correction, reducing overfitting, and pair naturally with `400` trees. |

Total combinations: `2 x 3 x 2 = 12` (well under 36).

In [4]:
# AI-GENERATED (Claude via Antigravity)
xgb_pipe = Pipeline([("prep", make_preprocessor(features)),
                     ("model", XGBRegressor(subsample=0.8, random_state=42, n_jobs=1))])
xgb_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.03, 0.1],
}
xgb_sample = Pipeline([("prep", make_preprocessor(features)),
                       ("model", XGBRegressor(n_estimators=300, max_depth=5, learning_rate=0.05, subsample=0.8, random_state=42, n_jobs=-1))])
gs_xgb, row_xgb = estimate_and_tune("XGBoost", "boosting", xgb_pipe, xgb_grid, xgb_sample)
tuned_rows.append(row_xgb); best_params["XGBoost"] = gs_xgb.best_params_

[XGBoost] 12 combinations x 5 folds = 60 fits.
Estimated runtime: ~15s (1 full fit took 1.8s)


,rank_test_score,mean_CV_neg_RMSE_log,std_CV,param_model__n_estimators,param_model__max_depth,param_model__learning_rate
0,1,-0.630882,0.047870,200,3,0.03
1,2,-0.630895,0.049620,400,3,0.03
2,3,-0.635020,0.048637,200,5,0.03
6,4,-0.636366,0.050497,200,3,0.10
3,5,-0.638949,0.050147,400,5,0.03


Best params: {'model__learning_rate': 0.03, 'model__max_depth': 3, 'model__n_estimators': 200}
At grid edge?: model__n_estimators=200 (edge of [200, 400]); model__max_depth=3 (edge of [3, 5, 7]); model__learning_rate=0.03 (edge of [0.03, 0.1])
Done in 53.8s -> train RMSE: 37.64 | test RMSE: 25.96 | test MAE: 11.91 | R2: 0.829


## 3. CatBoost (Boosting with native `district`)
Default CatBoost was the strongest of the three base models (`test RMSE 27.6`). We use `l2_leaf_reg=5.0` and tune tree count, depth, and learning rate.

| parameter | candidate values | why this range |
|---|---|---|
| `iterations` | `[300, 600]` | More trees lower bias, while fewer trees (`300`) stop early before memorising noise. |
| `depth` | `[4, 6, 8]` | Shallower symmetric trees (`4`) limit variance; deeper trees (`8`) lower bias on district-climate interactions. |
| `learning_rate` | `[0.03, 0.1]` | A smaller learning rate (`0.03`) needs more iterations and reduces overfitting. |

Total combinations: `2 x 3 x 2 = 12` (well under 36).

In [5]:
# AI-GENERATED (Claude via Antigravity)
cb_est = CatBoostDistrict(l2_leaf_reg=5.0, random_state=42, thread_count=1)
cb_grid = {
    "iterations": [300, 600],
    "depth": [4, 6, 8],
    "learning_rate": [0.03, 0.1],
}
cb_sample = CatBoostDistrict(iterations=450, depth=6, learning_rate=0.05, l2_leaf_reg=5.0, random_state=42, thread_count=-1)
gs_cb, row_cb = estimate_and_tune("CatBoost", "boosting", cb_est, cb_grid, cb_sample)
tuned_rows.append(row_cb); best_params["CatBoost"] = gs_cb.best_params_

[CatBoost] 12 combinations x 5 folds = 60 fits.
Estimated runtime: ~92s (1 full fit took 11.2s)


,rank_test_score,mean_CV_neg_RMSE_log,std_CV,param_iterations,param_depth,param_learning_rate
2,1,-0.626472,0.053916,600,4,0.03
0,2,-0.626940,0.052387,300,4,0.03
4,3,-0.632266,0.048964,300,6,0.03
6,4,-0.632671,0.049670,600,6,0.03
1,5,-0.634425,0.052991,300,4,0.10


Best params: {'depth': 4, 'iterations': 600, 'learning_rate': 0.03}
At grid edge?: iterations=600 (edge of [300, 600]); depth=4 (edge of [4, 6, 8]); learning_rate=0.03 (edge of [0.03, 0.1])
Done in 328.5s -> train RMSE: 27.98 | test RMSE: 25.92 | test MAE: 12.09 | R2: 0.829


## Save best parameters and compare all models
We save `models/best_params.json` and append the tuned models to the previous results table in `reports/results_tuned.csv`.

In [6]:
# AI-GENERATED (Claude via Antigravity)
os.makedirs("../models", exist_ok=True)
with open("../models/best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)
print("Saved ../models/best_params.json")

prev = pd.read_csv("../reports/results_default.csv").drop(columns=["test/train RMSE"], errors="ignore")
res = pd.concat([prev, pd.DataFrame(tuned_rows)], ignore_index=True).sort_values("test_RMSE").reset_index(drop=True)
res["test/train RMSE"] = res["test_RMSE"] / res["train_RMSE"]
res.to_csv("../reports/results_tuned.csv", index=False)
display(res.round(3))

Saved ../models/best_params.json


,model,family,train_RMSE,test_RMSE,test_MAE,test_R2,train_seconds,test/train RMSE
0,Random Forest (tuned),bagging,26.620,25.787,11.951,0.831,229.720,0.969
1,CatBoost (tuned),boosting,27.975,25.921,12.094,0.829,328.490,0.927
2,XGBoost (tuned),boosting,37.642,25.961,11.912,0.829,53.780,0.690
3,Gradient Boosting,boosting,27.213,26.054,12.035,0.827,25.182,0.957
4,Voting (RF + XGB + CatBoost),combined,16.880,26.797,12.231,0.817,43.485,1.588
5,"Stacking (RF + XGB + CatBoost, RidgeCV)",combined,14.795,27.282,12.509,0.811,191.420,1.844
6,CatBoost,boosting,21.955,27.613,12.645,0.806,13.074,1.258
7,Baseline: persistence (this week),baseline,28.157,27.677,12.802,0.805,0.000,0.983
8,Random Forest,bagging,11.533,27.821,12.484,0.803,27.480,2.412
9,Bagging (decision trees),bagging,11.746,27.878,12.525,0.802,25.251,2.373
